# Updated DeepSeek-V4-Flash ChildSnack (LLM planner, single-sandwich-line direct unraveling)

In [1]:
# Cell 1: Setup (DeepSeek planner pipeline, ChildSnack single-sandwich-line independent)
import os
import re
import json
import time
import subprocess
import platform
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
import requests

# Project root (run notebook from formalizer-scaling/src/planner_gen_notebooks/ChildSnack/single_sandwich_line)
PROJECT_ROOT = Path.cwd().parents[3]

# Data / prompt / output paths
DATA_DIR = PROJECT_ROOT / "data"
PDDL_DIR = DATA_DIR / "pddl_unraveling_domain" / "ChildSnack" / "single_sandwich_line"
DESC_DIR = DATA_DIR / "descriptions_unraveling_domain" / "ChildSnack" / "single_sandwich_line"

DOMAIN_PROMPT_PATH = PROJECT_ROOT / "prompts" / "ChildSnack" / "llm_planner_instruction.txt"
FALLBACK_PROMPT_PATH = PROJECT_ROOT / "prompts" / "BlocksWorld" / "llm_planner_instruction.txt"
PROMPT_PATH = DOMAIN_PROMPT_PATH if DOMAIN_PROMPT_PATH.exists() else FALLBACK_PROMPT_PATH

MODEL_DIRNAME = "updated_deepseek_v4_flash"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "llm_planner_direct_unraveling" / "ChildSnack" / "single_sandwich_line" / MODEL_DIRNAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Project-local VAL executable for the current operating system
VAL_BUILD_PLATFORM = "macos64" if platform.system() == "Darwin" else "linux64"
VAL_PATH = PROJECT_ROOT / "VAL" / "build" / VAL_BUILD_PLATFORM / "Release" / "bin" / "Validate"

# Model + API
MODEL_NAME = "deepseek-v4-flash"
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise ValueError(f"DEEPSEEK_API_KEY not found in {ENV_PATH}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("VAL exists:", VAL_PATH.exists())
print("PDDL dir exists:", PDDL_DIR.exists())
print("Description dir exists:", DESC_DIR.exists())
print("Prompt exists:", PROMPT_PATH.exists())
print("Model:", MODEL_NAME)

PROJECT_ROOT: /Users/owenjiang/Desktop/Drexel NLP/Scalability_project/ours/formalizer-scaling-main
OUTPUT_ROOT: /Users/owenjiang/Desktop/Drexel NLP/Scalability_project/ours/formalizer-scaling-main/output/llm_planner_direct_unraveling/ChildSnack/single_sandwich_line/updated_deepseek_v4_flash
VAL exists: True
PDDL dir exists: True
Description dir exists: True
Prompt exists: True
Model: deepseek-v4-flash


In [2]:
# Cell 2: Load problem IDs and planner instruction
import pandas as pd

problem_files = sorted(PDDL_DIR.glob("p*_problem.pddl"))
PROBLEM_IDS = [p.name.removesuffix("_problem.pddl") for p in problem_files]

PLANNER_INSTRUCTION = PROMPT_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
if PROBLEM_IDS:
    print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Planner prompt chars:", len(PLANNER_INSTRUCTION))
print("PDDL dir exists:", PDDL_DIR.exists())
print("Description dir exists:", DESC_DIR.exists())

Num problems: 120
First/last: p01 p99
Planner prompt chars: 1388
PDDL dir exists: True
Description dir exists: True


In [3]:
# Cell 3: Helper - load NL descriptions and ground-truth PDDL for one problem ID
def load_problem_artifacts(problem_id: str):
    domain_desc_path = DESC_DIR / f"{problem_id}_domain.txt"
    problem_desc_path = DESC_DIR / f"{problem_id}_problem_clean2.txt"
    gt_domain_pddl_path = PDDL_DIR / "domain.pddl"
    gt_problem_pddl_path = PDDL_DIR / f"{problem_id}_problem.pddl"

    if not domain_desc_path.exists():
        raise FileNotFoundError(f"Missing domain description: {domain_desc_path}")
    if not problem_desc_path.exists():
        raise FileNotFoundError(f"Missing problem description: {problem_desc_path}")
    if not gt_domain_pddl_path.exists():
        raise FileNotFoundError(f"Missing ground-truth domain PDDL: {gt_domain_pddl_path}")
    if not gt_problem_pddl_path.exists():
        raise FileNotFoundError(f"Missing ground-truth problem PDDL: {gt_problem_pddl_path}")

    return {
        "domain_desc": domain_desc_path.read_text(),
        "problem_desc": problem_desc_path.read_text(),
        "gt_domain_pddl_path": gt_domain_pddl_path,
        "gt_problem_pddl_path": gt_problem_pddl_path,
        "gt_domain_pddl_text": gt_domain_pddl_path.read_text(),
    }

In [4]:
# Cell 4: Helper - build planner prompt (NL domain + NL problem + GT domain PDDL)
def build_prompt(problem_id: str):
    artifacts = load_problem_artifacts(problem_id)

    return (
        PLANNER_INSTRUCTION
        + "\n\nYou are an expert automated planner.\n"
        "Create an executable plan that is consistent with all provided information.\n\n"
        "### Natural-language domain description\n"
        f"{artifacts['domain_desc']}\n\n"
        "### Natural-language problem description\n"
        f"{artifacts['problem_desc']}\n\n"
        "### Ground-truth domain PDDL\n"
        f"{artifacts['gt_domain_pddl_text']}\n\n"
        "Output only the final plan wrapped in <plan>...</plan>."
    )

In [5]:
# Cell 5: Helper - call DeepSeek for direct plan generation
def call_deepseek(prompt: str, temperature: float = 0.4, max_output_tokens: int = 220000):
    url = f"{DEEPSEEK_BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_output_tokens,
    }

    response = requests.post(url, headers=headers, json=payload, timeout=600)
    response.raise_for_status()
    data = response.json()
    message = data["choices"][0]["message"]
    response_text = (message.get("content") or "").strip()
    reasoning_text = (message.get("reasoning_content") or "").strip()
    return response_text, reasoning_text

In [6]:
# Cell 6: Helper - extract <plan>...</plan> block
def extract_plan(raw_text: str):
  if not raw_text:
      return ""

  m = re.search(r"<plan>(.*?)</plan>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if not m:
      return ""

  plan_text = m.group(1).strip()

  # Keep one action per non-empty line
  lines = [ln.strip() for ln in plan_text.splitlines() if ln.strip()]
  return "\n".join(lines)

In [7]:
# Cell 7: Helper - validate a plan with VAL (GT domain + GT problem)
def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  proc = subprocess.run(cmd, capture_output=True, text=True)

  out_text = (proc.stdout or "").strip()
  err_text = (proc.stderr or "").strip()
  merged = (out_text + ("\n" + err_text if err_text else "")).strip()

  val_ok = "Plan valid" in merged
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [8]:
# Cell 8: Helper - run one problem end-to-end (generate plan + VAL), persist outputs
def process_one_problem(problem_id: str):
    artifacts = load_problem_artifacts(problem_id)
    prompt = build_prompt(problem_id)
    raw_text, reasoning_text = call_deepseek(prompt)
    plan_text = extract_plan(raw_text)

    cand_dir = OUTPUT_ROOT / problem_id / "cand_00"
    cand_dir.mkdir(parents=True, exist_ok=True)

    raw_path = cand_dir / "raw.txt"
    reasoning_path = cand_dir / "reasoning.txt"
    plan_path = cand_dir / "plan.txt"
    val_log_path = cand_dir / "validator.log"
    err_path = cand_dir / "planner_error.txt"

    raw_path.write_text(raw_text or "")
    reasoning_path.write_text(reasoning_text or "")
    (cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])
    (cand_dir / "problem.pddl").write_text(artifacts["gt_problem_pddl_path"].read_text())

    if not plan_text:
        plan_path.write_text("")
        err_msg = "Empty or missing <plan>...</plan> output."
        err_path.write_text(err_msg)
        return {
            "problem": problem_id,
            "plan_found": False,
            "val_ok": False,
            "planner_error": err_msg,
        }

    plan_path.write_text(plan_text + ("" if plan_text.endswith("\n") else "\n"))

    val_result = run_val(artifacts["gt_domain_pddl_path"], artifacts["gt_problem_pddl_path"], plan_path)
    val_log_path.write_text(val_result.get("val_output", "") or "")

    return {
        "problem": problem_id,
        "plan_found": True,
        "val_ok": bool(val_result["val_ok"]),
        "planner_error": "",
    }

In [9]:
# Cell 9: Generate only (no VAL)
smoke_problem = "p01"
artifacts = load_problem_artifacts(smoke_problem)

prompt = build_prompt(smoke_problem)
raw_text, reasoning_text = call_deepseek(prompt)
plan_text = extract_plan(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "reasoning.txt").write_text(reasoning_text or "")
(cand_dir / "plan.txt").write_text((plan_text + "\n") if plan_text else "")
(cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])
(cand_dir / "problem.pddl").write_text(artifacts["gt_problem_pddl_path"].read_text())

print("Generation-only done.")
print("problem:", smoke_problem)
print("reasoning_chars:", len(reasoning_text or ""))
print("extracted_plan_chars:", len(plan_text or ""))
print("saved:", cand_dir)

Generation-only done.
problem: p01
reasoning_chars: 143522
extracted_plan_chars: 639
saved: /Users/owenjiang/Desktop/Drexel NLP/Scalability_project/ours/formalizer-scaling-main/output/llm_planner_direct_unraveling/ChildSnack/single_sandwich_line/updated_deepseek_v4_flash/p01/cand_00


In [10]:
# Cell 10: VAL only (no model call), concise output
import contextlib
import io

smoke_problem = "p01"
artifacts = load_problem_artifacts(smoke_problem)
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

plan_path = cand_dir / "plan.txt"
if not plan_path.exists():
    raise FileNotFoundError(f"Missing plan file: {plan_path}")

plan_text = plan_path.read_text().strip()
if not plan_text:
    summary = {
        "problem": smoke_problem,
        "plan_found": False,
        "val_ok": False,
        "planner_error": "Empty plan.txt (no actions).",
    }
    (cand_dir / "planner_error.txt").write_text(summary["planner_error"])
    print("Produced files:")
    print("- planner_error.txt")
    print("VAL result: not run (empty plan)")
    print("Summary:")
    print(json.dumps(summary, indent=2))
else:
    with contextlib.redirect_stdout(io.StringIO()):
        val_result = run_val(artifacts["gt_domain_pddl_path"], artifacts["gt_problem_pddl_path"], plan_path)

    (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

    summary = {
        "problem": smoke_problem,
        "plan_found": True,
        "val_ok": bool(val_result["val_ok"]),
        "planner_error": "",
    }

    print("Produced files:")
    print("- validator.log")
    print(f"VAL result: {'valid' if val_result.get('val_ok') else 'invalid'}")
    print("Summary:")
    print(json.dumps(summary, indent=2))

Produced files:
- validator.log
VAL result: invalid
Summary:
{
  "problem": "p01",
  "plan_found": true,
  "val_ok": false,
  "planner_error": ""
}


In [11]:
# Cell 11: Batch generate plans only (no VAL)
from tqdm.auto import tqdm

BATCH_PROBLEMS_GEN = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(2, 81)]

gen_rows = []

for pid in tqdm(BATCH_PROBLEMS_GEN, desc="Generating plans"):
    artifacts = load_problem_artifacts(pid)
    prompt = build_prompt(pid)
    raw_text, reasoning_text = call_deepseek(prompt)
    plan_text = extract_plan(raw_text)

    cand_dir = OUTPUT_ROOT / pid / "cand_00"
    cand_dir.mkdir(parents=True, exist_ok=True)

    (cand_dir / "raw.txt").write_text(raw_text or "")
    (cand_dir / "reasoning.txt").write_text(reasoning_text or "")
    (cand_dir / "plan.txt").write_text((plan_text + "\n") if plan_text else "")
    (cand_dir / "domain.pddl").write_text(artifacts["gt_domain_pddl_text"])
    (cand_dir / "problem.pddl").write_text(artifacts["gt_problem_pddl_path"].read_text())

    gen_rows.append(
        {
            "problem": pid,
            "plan_extracted": bool(plan_text),
            "reasoning_chars": len(reasoning_text or ""),
            "plan_chars": len(plan_text or ""),
            "cand_dir": str(cand_dir),
        }
    )

print("Batch generation done.")
print("Problems:", BATCH_PROBLEMS_GEN[0], "->", BATCH_PROBLEMS_GEN[-1], f"({len(BATCH_PROBLEMS_GEN)})")
print("Plans extracted:", sum(1 for r in gen_rows if r["plan_extracted"]), "/", len(gen_rows))

/opt/anaconda3/envs/formalizer-scaling/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating plans: 100%|██████████| 79/79 [6:25:38<00:00, 292.90s/it]  

Batch generation done.
Problems: p02 -> p80 (79)
Plans extracted: 79 / 79


In [20]:
# Cell 12: Batch VAL only (no model call)
from tqdm.auto import tqdm
import contextlib
import io

BATCH_PROBLEMS_VAL = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(71, 81)]

val_rows = []

for pid in tqdm(BATCH_PROBLEMS_VAL, desc="Running VAL"):
    artifacts = load_problem_artifacts(pid)
    cand_dir = OUTPUT_ROOT / pid / "cand_00"
    plan_path = cand_dir / "plan.txt"

    row = {
        "problem": pid,
        "plan_found": False,
        "val_ok": False,
        "planner_error": "",
    }

    if not plan_path.exists():
        row["planner_error"] = f"Missing plan file: {plan_path}"
        (cand_dir / "planner_error.txt").write_text(row["planner_error"])
        val_rows.append(row)
        continue

    plan_text = plan_path.read_text().strip()
    if not plan_text:
        row["planner_error"] = "Empty plan.txt (no actions)."
        (cand_dir / "planner_error.txt").write_text(row["planner_error"])
        val_rows.append(row)
        continue

    row["plan_found"] = True

    with contextlib.redirect_stdout(io.StringIO()):
        val_result = run_val(artifacts["gt_domain_pddl_path"], artifacts["gt_problem_pddl_path"], plan_path)

    (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
    row["val_ok"] = bool(val_result["val_ok"])

    val_rows.append(row)

print("Batch VAL done.")
print("Problems:", BATCH_PROBLEMS_VAL[0], "->", BATCH_PROBLEMS_VAL[-1], f"({len(BATCH_PROBLEMS_VAL)})")
print("Plan found:", sum(int(r["plan_found"]) for r in val_rows), "/", len(val_rows))
print("VAL valid:", sum(int(r["val_ok"]) for r in val_rows), "/", len(val_rows))

Running VAL: 100%|██████████| 10/10 [00:00<00:00, 22.91it/s]

Batch VAL done.
Problems: p71 -> p80 (10)
Plan found: 10 / 10
VAL valid: 0 / 10


In [ ]:
# Cell 13: In-notebook metrics summary (no persistence)
import pandas as pd

# Independent handle for any selected batch
SUMMARY_PROBLEMS = [f"p{i:02d}" for i in range(1, 11)]  # example: p01..p10

rows = []
for pid in SUMMARY_PROBLEMS:
  cand_dir = OUTPUT_ROOT / pid / "cand_00"

  raw_path = cand_dir / "raw.txt"
  plan_path = cand_dir / "plan.txt"
  val_log_path = cand_dir / "validator.log"
  err_path = cand_dir / "planner_error.txt"

  raw_text = raw_path.read_text() if raw_path.exists() else ""
  plan_text = plan_path.read_text() if plan_path.exists() else ""
  val_log = val_log_path.read_text() if val_log_path.exists() else ""
  planner_error = err_path.read_text().strip() if err_path.exists() else ""

  plan_found = bool(plan_text.strip())
  val_ran = bool(val_log.strip())
  val_ok = "Plan valid" in val_log

  if val_ok:
      status = "plan_valid"
  elif plan_found and val_ran:
      status = "plan_invalid"
  else:
      status = "planner_failed"

  rows.append(
      {
          "problem_id": pid,
          "status": status,
          "plan_found": plan_found,
          "val_ran": val_ran,
          "val_ok": val_ok,
          "planner_error": planner_error,
          "raw_chars": len(raw_text),
          "plan_chars": len(plan_text),
          "cand_dir": str(cand_dir),
      }
  )

details_df = pd.DataFrame(rows).sort_values("problem_id").reset_index(drop=True)

total = len(details_df)
plan_found_n = int(details_df["plan_found"].sum())
val_ok_n = int(details_df["val_ok"].sum())
planner_failed_n = int((details_df["status"] == "planner_failed").sum())
plan_invalid_n = int((details_df["status"] == "plan_invalid").sum())

summary_df = pd.DataFrame(
  [
      {
          "num_problems": total,
          "plan_found_count": plan_found_n,
          "plan_found_rate": plan_found_n / total if total else 0.0,
          "val_ok_count": val_ok_n,
          "val_ok_rate": val_ok_n / total if total else 0.0,
          "planner_failed_count": planner_failed_n,
          "plan_invalid_count": plan_invalid_n,
      }
  ]
)

print("Summary:")
display(summary_df)

print("Per-problem details:")
display(details_df[["problem_id", "status", "plan_found", "val_ran", "val_ok", "planner_error"]])

# Keep in memory for Cell 14
SUMMARY_DF = summary_df
DETAILS_DF = details_df
SUMMARY_PROBLEMS_USED = SUMMARY_PROBLEMS

In [ ]:
# Cell 14: Persist summary/details to CSV (run only when you want to save)
from datetime import datetime

SAVE_LABEL = "updated_deepseek_v4_flash_llm_planner_direct_unraveling_childsnack"

METRIC_DIR = OUTPUT_ROOT / "_metrics"
METRIC_DIR.mkdir(parents=True, exist_ok=True)

if "SUMMARY_DF" not in globals() or "DETAILS_DF" not in globals():
    raise RuntimeError("Run Cell 13 first to populate SUMMARY_DF and DETAILS_DF.")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
problems_tag = (
    f"{SUMMARY_PROBLEMS_USED[0]}_to_{SUMMARY_PROBLEMS_USED[-1]}"
    if len(SUMMARY_PROBLEMS_USED) > 1
    else SUMMARY_PROBLEMS_USED[0]
)

summary_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_summary_{ts}.csv"
details_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_details_{ts}.csv"

SUMMARY_DF.to_csv(summary_out, index=False)
DETAILS_DF.to_csv(details_out, index=False)

print("Saved:")
print("-", summary_out)
print("-", details_out)
